#### ALL SAMPLES COLLECTED BEFORE QUESTION 1.4 IS ADDED TO THE eCRF

In [ ]:
import os
from dotenv import load_dotenv
import redcap
import pandas as pd
import pandasql as psql
from datetime import datetime, timedelta
import re
from sklearn.metrics.pairwise import cosine_similarity
from redcap import Project
today=pd.Timestamp.today()

# Load environment variables
load_dotenv()

# Initialize REDCap projects
df = redcap.Project(
    os.getenv('REDCAP_MAIN_URL'),
    os.getenv('REDCAP_MAIN_TOKEN')
)

sen_project = redcap.Project(
    os.getenv('REDCAP_CK_WK_URL'),
    os.getenv('REDCAP_CK_WK_TOKEN')
)

project = redcap.Project(
    os.getenv('REDCAP_INDIGO_URL'),
    os.getenv('REDCAP_INDIGO_TOKEN')
)

In [9]:
#extract study sample processing data
BLD_sample_processing = project.export_records(forms=['blood_sample_processing'])

#convert data to Dataframe.
sp_data=pd.DataFrame(BLD_sample_processing)

In [10]:
import pandas as pd

# Specify the fields you want to extract based on logics
SP_data = sp_data[['con_participantid_q1', 'redcap_event_name', 'bsp_date', 'bsp_time', 'bsp_rbc', 'blood_sample_processing_complete']]
SP_data = pd.DataFrame(SP_data)

# Convert bsp_date and bsp_time to appropriate data types
SP_data['bsp_date'] = pd.to_datetime(SP_data['bsp_date'], errors='coerce').dt.date
SP_data['bsp_time'] = pd.to_datetime(SP_data['bsp_time'], errors='coerce').dt.time

# Define the conditions
condition_arm_1 = ((SP_data['redcap_event_name'] == 'week_20_arm_1') |
                   (SP_data['redcap_event_name'] == 'week_28_arm_1') |
                   (SP_data['redcap_event_name'] == 'week_36_arm_1') |
                   (SP_data['redcap_event_name'] == 'month_1_arm_1') |
                   (SP_data['redcap_event_name'] == 'month_6_arm_1') |
                   (SP_data['redcap_event_name'] == 'month_12_arm_1'))

condition_arm_2 = ((SP_data['redcap_event_name'] == 'month_1_arm_2') |
                   (SP_data['redcap_event_name'] == 'month_6_arm_2') |
                   (SP_data['redcap_event_name'] == 'month_12_arm_2'))

blood_processing_condition = (SP_data['bsp_rbc'].isnull()) | (SP_data['bsp_rbc'] == "")

# Filter the data
df_SP = SP_data[(condition_arm_1 & blood_processing_condition) | (condition_arm_2 & blood_processing_condition)]

df_SP = pd.DataFrame(df_SP)


In [11]:
#Save the filtered DataFrame to a CSV file
df_SP.to_csv('Redblood cell blnk.csv', index=False)